# 🧠 CalRetail — Dynamic Pricing
## Goal
Optimise margins and stock turnover using each product's own real price elasticity, real cost
price, and real inventory/demand signals — benchmarked against competitor pricing.

## Algorithmic Explanation
**Multi-Factor Scoring Optimizer, driven by per-SKU elasticity**
1. Extract current price, real cost price, and competitor average/minimum pricing.
2. Compute inventory ratio and stockout risk from real inventory data, benchmarked against the
   population's own median inventory ratio (not an arbitrary flat target).
3. Use each product's real daily sales velocity (`feature_inventory_health.avg_daily_demand`)
   and real price elasticity (`adaptive_thresholds.get_price_elasticity`, regressed from this
   SKU's own pricing history) to size the adjustment and the expected revenue lift.
4. Cap outputs between the real cost-price margin floor and a maximum limit.



In [ ]:
import os
import sys
import numpy as np
import pandas as pd
from pathlib import Path
import warnings
import json
import re
import math
from backend.utils.db import load_table  # SQLite-backed
warnings.filterwarnings('ignore')

# Set path to include parent directory
base_path = Path().resolve()
while not (base_path / 'data').exists() and base_path.parent != base_path:
    base_path = base_path.parent
processed_dir = base_path / 'data'
if str(base_path) not in sys.path:
    sys.path.insert(0, str(base_path))

print(f"Project root found at: {base_path}")
print(f"Data directory: {processed_dir}")

In [ ]:
from backend.utils.adaptive_thresholds import get_price_elasticity

cust_pr    = load_table('competitor_pricing')
prod       = load_table('products')
inv        = load_table('inventory')
inv_health = load_table('feature_inventory_health')

# Population reference points, used instead of an arbitrary flat 0.45 "ideal"
# inventory ratio and a fixed velocity offset.
_inv_by_product = inv.groupby('product_id').agg(stock_qty=('stock_qty', 'sum'), max_stock=('max_stock', 'sum'))
_inv_by_product = _inv_by_product[_inv_by_product['max_stock'] > 0]
median_inventory_ratio = float((_inv_by_product['stock_qty'] / _inv_by_product['max_stock']).median())
median_daily_demand = float(inv_health['avg_daily_demand'].median())

print(f"Competitor price samples: {cust_pr.shape}")
print(f"Active items: {prod.shape}")
print(f"Population median inventory ratio: {median_inventory_ratio:.2f} | median daily demand: {median_daily_demand:.2f}")

In [ ]:
def recommend_dynamic_price(product_id):
    p_info = prod[prod['product_id'] == product_id].iloc[0]
    current_price = float(p_info['price'])
    # Real cost price from the product catalogue, not a guessed 0.7x margin.
    cost_price = float(p_info.get('cost_price', current_price * 0.6)) or current_price * 0.6

    # Calculate competitor metrics
    comp_matches = cust_pr[cust_pr['product_id'] == product_id]
    if not comp_matches.empty:
        comp_min = float(comp_matches['price'].min())
        comp_max = float(comp_matches['price'].max())
        comp_avg = float(comp_matches['price'].mean())
    else:
        comp_min = current_price * 0.85
        comp_max = current_price * 1.15
        comp_avg = current_price

    # Calculate inventory metrics
    prod_inv = inv[inv['product_id'] == product_id]
    stock_qty = float(prod_inv['stock_qty'].sum())
    max_qty = float(prod_inv['max_stock'].sum())
    if max_qty <= 0:
        max_qty = 500.0

    inventory_ratio = stock_qty / max_qty
    stockout_risk = float(prod_inv['stockout_risk'].max()) if not prod_inv.empty else 0.0

    # Real daily sales velocity (units/day) from the engineered inventory
    # health table — not a cumulative lifetime total mislabeled as a "rate".
    health_match = inv_health[inv_health['product_id'] == product_id]
    sales_velocity = float(health_match['avg_daily_demand'].mean()) if not health_match.empty else median_daily_demand

    # Real, product-specific price elasticity — a regression of quantity
    # change vs. price change on this SKU's own pricing history — replacing a
    # fixed -1.4 guess applied to every product regardless of category or price point.
    elasticity = get_price_elasticity(product_id)

    # Continuous inventory adjustment, benchmarked against the *population's*
    # real median inventory ratio rather than an arbitrary flat target.
    inv_factor = 0.14 * (median_inventory_ratio - inventory_ratio)

    # Stockout risk premium
    risk_factor = 0.08 * stockout_risk

    # Sales velocity premium, benchmarked against the population's real
    # median daily demand rather than a fixed log-offset constant.
    vel_factor = float(np.clip(
        0.04 * (math.log1p(sales_velocity) - math.log1p(max(median_daily_demand, 0.1))),
        -0.06, 0.06
    ))

    total_adj = inv_factor + risk_factor + vel_factor

    # Base recommendation on competitor average modified by our adjustment factors
    recommended = comp_avg * (1.0 + total_adj)

    # Ensure recommended price is within 25% of current price to avoid wild jumps
    recommended = max(current_price * 0.75, min(recommended, current_price * 1.25))

    # Ensure recommended price stays above a real margin floor over actual cost
    recommended = max(cost_price * 1.05, recommended)

    # Calculate price delta
    price_delta_pct = ((recommended - current_price) / current_price) * 100.0

    # Dynamic expected revenue lift using this SKU's own real elasticity
    volume_lift_pct = elasticity * price_delta_pct
    revenue_lift_est = price_delta_pct + volume_lift_pct + (price_delta_pct * volume_lift_pct / 100.0)
    revenue_lift_est = max(0.5, round(revenue_lift_est, 2))

    # Generate unique rationale based on factors
    if inventory_ratio > 0.8:
        inventory_msg = f"Surplus stock (ratio {inventory_ratio:.1%} vs. population median {median_inventory_ratio:.1%}). Applied markdown to accelerate inventory velocity."
    elif inventory_ratio < 0.25:
        inventory_msg = f"Low stock alert (ratio {inventory_ratio:.1%} vs. population median {median_inventory_ratio:.1%}). Applied premium markup for margin optimization."
    else:
        inventory_msg = "Stock level stable. Pricing optimized against competitor average."

    return {
        "product_id": product_id,
        "product_name": p_info['product_name'],
        "current_price": round(float(current_price), 2),
        "competitor_avg": round(float(comp_avg), 2),
        "recommended_price": round(float(recommended), 2),
        "stock_level": int(stock_qty),
        "inventory_nudge": inventory_msg,
        "est_revenue_lift_pct": round(float(revenue_lift_est), 2),
        "price_elasticity": round(float(elasticity), 2),
    }

first_prod_id = prod['product_id'].iloc[0]
backend_res = recommend_dynamic_price(first_prod_id)

In [ ]:
print("=== CALRETAIL SMART PRICING ENGINE ===")
print(f"Product: {backend_res['product_name']} ({backend_res['product_id']})")
print(f"Current Store Price: ₹{backend_res['current_price']} | Competitor Market Average: ₹{backend_res['competitor_avg']}")
print(f"Stock Volume: {backend_res['stock_level']} ({backend_res['inventory_nudge']})")
print(f"--> RECOMMENDED NEW PRICE: ₹{backend_res['recommended_price']} (Estimated Revenue Lift: {backend_res['est_revenue_lift_pct']}%)")
